# DJIA Four-Asset Library Walkthrough

This notebook is a compact end-to-end demo of the cleaned library API. It uses four DJIA names with a shared 2019 construction window and a 2020 out-of-sample evaluation window.


## What This Covers

- loading market data from a saved local Yahoo-style snapshot
- creating a `Universe` with explicit construction dates
- building multiple portfolio constructors on the same asset set
- inspecting in-sample metrics and HRP diagnostics
- running backtests and Monte Carlo evaluation
- plotting with `PortfolioVisualizer`
- saving the standard library output structure under `outputs/runs/`

This is meant as a library walkthrough rather than a thesis-results notebook, so it uses the standard reusable library surfaces instead of the larger thesis experiment runner.


In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "portafolios").exists() and (candidate / "notebooks").exists():
            return candidate
    raise RuntimeError("Could not locate the repository root from the current working directory.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import json

import pandas as pd
from IPython.display import display

from portafolios import (
    Universe,
    local_loader,
    EqualWeightConstructor,
    Markowitz,
    NaiveRiskParity,
    HRPStyle,
    HRPRecursive,
    Backtester,
    MonteCarloEngine,
    PortfolioVisualizer,
)

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

PROJECT_ROOT


In [ ]:
DATA_PATH = PROJECT_ROOT / "outputs" / "final_experimental_setup" / "data" / "yfinance_final_setup_raw.csv"
TICKERS = ["AAPL", "MSFT", "JPM", "V"]

START = "2019-01-02"
END = "2020-12-31"
CONSTRUCTION_START = "2019-01-02"
CONSTRUCTION_END = "2019-12-31"
BACKTEST_START = "2020-01-02"
BACKTEST_END = "2020-12-31"

MC_SIMULATIONS = 300
MC_SEED = 42
MC_INITIAL_VALUE = 1.0
MAX_MC_PATHS_TO_SAVE = 60

RUN_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "runs"
UNIVERSE_NAME = "djia_four_asset_library_walkthrough"

config = pd.Series(
    {
        "data_path": str(DATA_PATH.relative_to(PROJECT_ROOT)),
        "tickers": TICKERS,
        "start": START,
        "end": END,
        "construction_start": CONSTRUCTION_START,
        "construction_end": CONSTRUCTION_END,
        "backtest_start": BACKTEST_START,
        "backtest_end": BACKTEST_END,
        "mc_simulations": MC_SIMULATIONS,
        "mc_seed": MC_SEED,
        "run_output_dir": str(RUN_OUTPUT_DIR.relative_to(PROJECT_ROOT)),
    },
    name="demo_config",
)

display(config)


## Build The Universe

This uses the cleaned public API and a saved local snapshot, so the walkthrough is fully reproducible without a live download.


In [ ]:
universe = Universe(
    universe_name=UNIVERSE_NAME,
    loader=local_loader,
    loader_kwargs={"path": DATA_PATH, "prefer_adj_close": True},
    tickers=TICKERS,
    start=START,
    end=END,
    construction_start=CONSTRUCTION_START,
    construction_end=CONSTRUCTION_END,
    base_output_dir=RUN_OUTPUT_DIR,
    auto_save_data=False,
).prepare_data()

run_dir = universe.output_dir
run_dir


In [ ]:
overview = pd.DataFrame(
    {
        "value": [
            universe.prices.index.min().date(),
            universe.prices.index.max().date(),
            len(universe.prices),
            len(universe.returns),
            list(universe.prices.columns),
            universe.construction_start.date(),
            universe.construction_end.date(),
            str(run_dir.relative_to(PROJECT_ROOT)),
        ]
    },
    index=[
        "price_start",
        "price_end",
        "n_price_rows",
        "n_return_rows",
        "tickers",
        "construction_start",
        "construction_end",
        "output_dir",
    ],
)

display(overview)
display(universe.prices.head())
display(universe.returns.head())


## Build Constructors On The Same Universe

This is the standard comparison workflow: all methods use the same assets and the same construction window.


In [ ]:
universe.build(EqualWeightConstructor())
universe.build(Markowitz(), ret_kind="simple", allow_short=False, on_failure="use_initial_weights")
universe.build(NaiveRiskParity())
universe.build(HRPStyle(n_clusters=2, display_name="HRP Style (2 Clusters)"))
universe.build(HRPRecursive())

universe.list_constructions()


In [ ]:
construction_summary = pd.DataFrame(
    [
        {
            "name": name,
            "display_name": result.display_name,
            "method_id": result.method_id,
            "n_selected": result.metrics_insample.get("n_selected"),
            "expected_return": result.metrics_insample.get("expected_return"),
            "volatility": result.metrics_insample.get("volatility"),
            "sharpe_m": result.metrics_insample.get("sharpe_m"),
        }
        for name, result in universe.constructions.items()
    ]
).set_index("name")

weights_table = pd.DataFrame(
    {
        name: universe.get_construction(name).weights
        for name in universe.list_constructions()
    }
).fillna(0.0)

display(construction_summary.sort_index())
display(weights_table)


In [ ]:
hrp_result = universe.get_construction("hrp_style")
hrp_diag = hrp_result.hrp_diagnostics

display(pd.Series(hrp_diag.cluster_assets, name="assets_in_cluster") if hrp_diag is not None else pd.Series(dtype=object))
display(universe.get_hrp_dist_matrix("hrp_style"))


## Run Evaluation

For this walkthrough we pass explicit 2020 dates into `Backtester.run_all(...)`. If you omit them, `Backtester.run()` defaults to the first available date after `construction_end` through the last available date in the universe.


In [ ]:
backtest_results = Backtester.run_all(
    universe,
    start_date=BACKTEST_START,
    end_date=BACKTEST_END,
    ann_factor=252,
    attach=True,
)

backtest_summary = pd.DataFrame(
    [
        {"name": name, **result.summary_metrics}
        for name, result in backtest_results.items()
    ]
).set_index("name")

display(backtest_summary.sort_values("sharpe_ratio", ascending=False))


In [ ]:
markowitz_window_summary = Backtester(universe, "markowitz_max_sharpe").summarize_window(
    result=backtest_results["markowitz_max_sharpe"],
    start_date="2020-03-01",
    end_date="2020-12-31",
)

pd.Series(markowitz_window_summary, name="markowitz_window_summary")


In [ ]:
mc_horizon = len(universe.get_returns_window(BACKTEST_START, BACKTEST_END))
mc_results = MonteCarloEngine.run_all(
    universe,
    horizon=mc_horizon,
    n_simulations=MC_SIMULATIONS,
    initial_value=MC_INITIAL_VALUE,
    seed=MC_SEED,
    attach=True,
)

mc_summary = pd.DataFrame(
    [
        {"name": name, **result.summary_metrics}
        for name, result in mc_results.items()
    ]
).set_index("name")

display(pd.Series({"mc_horizon": mc_horizon, "n_simulations": MC_SIMULATIONS, "seed": MC_SEED}, name="mc_config"))
display(mc_summary.sort_values("mean_terminal_value", ascending=False))


## Plot With `PortfolioVisualizer`

The visualizer is the cleaned plotting layer. The next cell shows representative figures from the three main families: backtests, constructions, and Monte Carlo.


In [ ]:
visualizer = PortfolioVisualizer(universe)

display(visualizer.plot_backtest_comparison())
display(visualizer.plot_backtest("hrp_recursive"))
display(visualizer.plot_weights_bar("markowitz_max_sharpe"))
display(visualizer.plot_mc_distribution("hrp_style"))


## Save The Standard Library Output Structure

This writes the reusable framework layout under `outputs/runs/<universe_name>/`. In the cleaned structure, numeric artifacts stay in `data/`, `constructions/`, `backtests/`, and `monte_carlo/`, while HTML figures live under `plots/`.


In [ ]:
saved = visualizer.save_everything(max_mc_paths=MAX_MC_PATHS_TO_SAVE)

saved_summary = pd.Series(
    {
        "market_data_dir": str(saved["market_data_dir"].relative_to(PROJECT_ROOT)),
        "n_constructions_saved": len(saved["constructions"]),
        "n_backtests_saved": len(saved["backtests"]),
        "n_monte_carlo_saved": len(saved["monte_carlo"]),
        "construction_plot_groups": list(saved["construction_plots"].keys()),
        "backtest_plot_groups": list(saved["backtest_plots"].keys()),
        "monte_carlo_plot_groups": list(saved["monte_carlo_plots"].keys()),
    },
    name="saved_summary",
)

display(saved_summary)


In [ ]:
example_structure = {
    "data": sorted(str(path.relative_to(run_dir)) for path in (run_dir / "data").glob("*")),
    "construction_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "constructions" / "markowitz_max_sharpe").glob("*")
    ),
    "backtest_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "backtests" / "markowitz_max_sharpe").glob("*")
    ),
    "monte_carlo_example": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "monte_carlo" / "markowitz_max_sharpe").glob("*")
    ),
    "comparison_data": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "monte_carlo").glob("comparison_*.csv")
    ),
    "plot_examples": sorted(
        str(path.relative_to(run_dir))
        for path in (run_dir / "plots").rglob("*.html")
    )[:12],
}

print(json.dumps(example_structure, indent=2))


## Next Step

This notebook is a compact reference for the cleaned library workflow. For the larger thesis-scale experiment grid, use [scripts/run_final_experimental_setup.py](../../../scripts/run_final_experimental_setup.py).
